# 🤖 BERT for Sentiment Analysis
**State-of-the-Art NLP with Hugging Face Transformers**
---

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from transformers import pipeline

print(f'PyTorch version    : {torch.__version__}')
print(f'Transformers version: {transformers.__version__}')
print('Libraries loaded ✅')

## 2. Load Dataset
> We use a nuanced synthetic dataset containing phrases with negations (e.g., "not bad") to demonstrate BERT's superior contextual understanding compared to bag-of-words models.

In [ ]:
df = pd.read_csv('../data/reviews.csv')
print(f'Shape   : {df.shape}')
print(f'Classes : {df["label"].value_counts().to_dict()}')
df.head()

## 3. The BERT Tokenizer (WordPiece)
> BERT does not read raw words. It uses **WordPiece** tokenization, which breaks words into subwords. This allows the model to handle out-of-vocabulary words gracefully by breaking them into known chunks (e.g., "unbelievable" → "un", "##believ", "##able").

In [ ]:
# Using a robust, state-of-the-art RoBERTa model for highly accurate predictions
model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(model_name)

sample_text = "The product is not bad at all, I actually love it!"
print(f"Original: {sample_text}\n")

# Tokenize
tokens = tokenizer.tokenize(sample_text)
print(f"Tokens: {tokens}\n")

# Encode to IDs (adds <s> and </s> for RoBERTa)
input_ids = tokenizer.encode(sample_text, add_special_tokens=True)
print(f"Input IDs: {input_ids}\n")

# Decode back to verify
decoded = tokenizer.decode(input_ids)
print(f"Decoded: {decoded}")

## 4. Zero-Shot Inference with Pre-trained BERT
> Hugging Face provides a simple `pipeline` API to run inference on state-of-the-art models without writing the training loop ourselves. This model was pre-trained on massive text corpora and fine-tuned on the SST-2 sentiment dataset.

In [ ]:
classifier = pipeline("sentiment-analysis", model=model_name, framework="pt")

test_sentences = [
    "The product is not bad at all, I actually love it!",
    "The movie was not good, I hated every minute of it.",
    "Absolutely brilliant, exceeded all my expectations!",
    "Fast delivery, but the item was completely broken."
]

print("--- Robust Model Predictions ---")
for sentence in test_sentences:
    result = classifier(sentence)[0]
    label = result['label'].capitalize() # 'Positive', 'Negative', or 'Neutral'
    print(f"[{label:^10}] (Confidence: {result['score']:.4f}) | {sentence}")

## 5. Evaluating on Our Dataset

In [ ]:
# Run inference on the entire dataset
texts = df['text'].tolist()
true_labels = df['sentiment'].tolist()

# Process in batches for efficiency
predictions = classifier(texts, batch_size=32, truncation=True, max_length=512)

# Convert predictions to binary (0 or 1). 'positive' -> 1, else -> 0
pred_labels = [1 if p['label'].lower() == 'positive' else 0 for p in predictions]

# Calculate metrics
acc = accuracy_score(true_labels, pred_labels)
print(f'\\nOverall Accuracy: {acc:.4f}')
print('\\nClassification Report:')
print(classification_report(true_labels, pred_labels, target_names=['Negative (0)', 'Positive (1)']))

# Confusion Matrix
cm = confusion_matrix(true_labels, pred_labels)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', ax=ax,
            xticklabels=['Negative (0)', 'Positive (1)'],
            yticklabels=['Negative (0)', 'Positive (1)'],
            linewidths=1, linecolor='white')
ax.set_title('BERT Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 6. How to Fine-Tune BERT on Custom Data
> While the pre-trained model works well out-of-the-box, you can fine-tune it on your specific domain data using the Hugging Face `Trainer` API. Below is the standard template for fine-tuning.

In [ ]:
# 1. Prepare dataset
from datasets import Dataset

def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

# Create Hugging Face Dataset
hf_dataset = Dataset.from_pandas(df[['text', 'sentiment']])
hf_dataset = hf_dataset.rename_column('sentiment', 'label') # Trainer expects 'label'

# Tokenize
tokenized_dataset = hf_dataset.map(tokenize_function, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(['text'])
tokenized_dataset.set_format('torch')

# Split
split_dataset = tokenized_dataset.train_test_split(test_size=0.2, seed=42)

print(f"Training samples: {len(split_dataset['train'])}")
print(f"Eval samples: {len(split_dataset['test'])}")

In [ ]:
# 2. Load model for fine-tuning
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 3. Define training arguments
training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
)

# 4. Define compute_metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    return {'accuracy': acc}

# 5. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split_dataset['train'],
    eval_dataset=split_dataset['test'],
    compute_metrics=compute_metrics,
)

# 6. Train (Uncomment to run)
# trainer.train()
# trainer.evaluate()

print("Fine-tuning template ready. Uncomment trainer.train() to execute.")

## 7. Save Fine-Tuned Model

In [ ]:
# After training, save the model and tokenizer
# model.save_pretrained('../models/fine_tuned_bert')
# tokenizer.save_pretrained('../models/fine_tuned_bert')
print("Model saving code ready.")

## 8. Key Takeaways
> - **WordPiece Tokenization**: Handles unknown words by breaking them into subwords, reducing the OOV problem.
> - **Self-Attention**: Allows the model to weigh the importance of all words in a sentence relative to each other, capturing complex negations and long-range dependencies.
> - **Transfer Learning**: Pre-training on massive corpora (like Wikipedia + BookCorpus) gives BERT a deep understanding of language, requiring only minimal fine-tuning for specific tasks.
> - **Special Tokens**: `[CLS]` aggregates sequence meaning for classification; `[SEP]` separates segments; `[MASK]` enables self-supervised pre-training.